In [0]:
"""
sim_crm_delta.py
================
Simulation script: generates synthetic CRM data and writes it into
Databricks Delta tables under  smart_odoo.silver

Tables covered (7 total):
    1. silver.crm_stage          ← fixed reference; seeded once, skipped on re-runs
    2. silver.crm_team           ← fixed reference; seeded once, skipped on re-runs
    3. silver.crm_lost_reason    ← fixed reference; seeded once, skipped on re-runs
    4. silver.utm_campaign       ← fixed reference; seeded once, skipped on re-runs
    5. silver.utm_source         ← fixed reference; seeded once, skipped on re-runs
    6. silver.utm_medium         ← fixed reference; seeded once, skipped on re-runs
    7. silver.crm_lead           ← 300 NEW rows on every run

════════════════════════════════════════════════════════════
SIMULATION RULES
════════════════════════════════════════════════════════════

[GENERAL]
- Catalog / Schema   : smart_odoo.silver
- Source tag         : dwh_source_db = "python_sim"
- Date range         : 2025-01-01  →  2026-04-22
- All IDs            : MAX(existing_id) + 1  →  append-safe, zero duplicates
- company_id         : always 1  ("Smart Odoo LLC")
- partner_id         : 1 – 50
- user_id            : 1 – 10

[REFERENCE TABLES]  crm_stage / crm_team / crm_lost_reason /
                    utm_campaign / utm_source / utm_medium
- Fixed seed data loaded ONLY on first run (skip if name already exists)
- These tables are dimension tables; their IDs are stable across runs
- crm_stage (5)    : New | Qualified | Proposition | Won | Lost
- crm_team (3)     : Direct Sales | Online Sales | Field Sales
- crm_lost_reason (4): Price | No budget | No need | Other
- utm_campaign (4) : Summer Sale | Q1 Awareness | Retention | Launch
- utm_source (5)   : Google | Facebook | LinkedIn | Email | Referral
- utm_medium (4)   : CPC | Organic | Social | Email

[CRM LEADS]  silver.crm_lead  — 300 per run
- type             : lead (40 %) | opportunity (60 %)
- priority         : 0=normal (50 %) | 1=low (25 %) | 2=high (15 %) | 3=urgent (10 %)
- stage progression logic:
    New        → probability  5–20 %,  won_status = "pending"
    Qualified  → probability 20–40 %,  won_status = "pending"
    Proposition→ probability 40–70 %,  won_status = "pending"
    Won        → probability 90–100 %, won_status = "won",  date_closed = date_open + 5–60d
    Lost       → probability  0–15 %,  won_status = "lost", date_closed = date_open + 5–60d,
                                        lost_reason_id populated
- active           : False only when won_status = "lost" (~70 % of lost leads)
- expected_revenue : 1,000 – 100,000 EGP
- prorated_revenue : expected_revenue × probability
- recurring_revenue: 0 or small fraction of expected_revenue (~20 % of leads)
- day_open / day_close : derived from date_open / date_closed
- date_deadline    : date_open + 10–90 days (nullable 25 %)
- date_last_stage_update: date_open + 0–15 days
- date_conversion  : only for opportunity type (nullable 40 %)
- iap_enrich_done  : True ~30 % of leads
- phone_state      : correct | incorrect | None (weighted)
- email_state      : correct | incorrect | None (weighted)
- city             : Cairo | Alex | Giza | Mansoura | Tanta | Aswan (weighted to Egypt cities)
- country_id       : always 1  (Egypt)

════════════════════════════════════════════════════════════
"""

import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType,
    BooleanType, TimestampType
)

# ─────────────────────────────────────────────────────────────
# SPARK + CATALOG
# ─────────────────────────────────────────────────────────────
spark = SparkSession.builder.appName("sim_crm_delta").getOrCreate()

CATALOG    = "smart_odoo"
SCHEMA     = "silver"
SOURCE_DB  = "python_sim"
COMPANY_ID = 1
COMPANY    = "Smart Odoo LLC"
COUNTRY_ID = 1
COUNTRY    = "Egypt"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ─────────────────────────────────────────────────────────────
# DATE HELPERS
# ─────────────────────────────────────────────────────────────
START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 4, 22)

def rand_date() -> datetime:
    return START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )

def maybe(val, prob_none: float = 0.25):
    return None if random.random() < prob_none else val

now = datetime.now()

# ─────────────────────────────────────────────────────────────
# HELPER: safe append — skip existing names in dimension tables
# ─────────────────────────────────────────────────────────────
def existing_names(table: str, name_col: str = "name") -> set:
    try:
        return {
            r[name_col] for r in
            spark.sql(f"SELECT {name_col} FROM {CATALOG}.{SCHEMA}.{table}").collect()
        }
    except Exception:
        return set()

def max_id(table: str, col: str) -> int:
    try:
        return spark.sql(
            f"SELECT COALESCE(MAX({col}), 0) FROM {CATALOG}.{SCHEMA}.{table}"
        ).collect()[0][0]
    except Exception:
        return 0

def append_delta(df, table: str):
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")
    )
    print(f"  ✅  {table}  (+{df.count()} rows)")


# ═══════════════════════════════════════════════════════════════
# 1. CRM STAGE  (5 fixed stages — seed once)
# ═══════════════════════════════════════════════════════════════
STAGE_SEED = [
    (1, 1,  "New",          None,  False, False, 30),
    (2, 2,  "Qualified",    None,  False, False, 20),
    (3, 3,  "Proposition",  None,  False, False, 15),
    (4, 4,  "Won",          None,  True,  True,  None),
    (5, 5,  "Lost",         None,  False, True,  None),
]
# (id, seq, name, requirements, is_won, fold, rotting_days)

STAGES = {row[0]: row[2] for row in STAGE_SEED}   # id → name

existing_stage_names = existing_names("crm_stage")
stage_rows = []
for sid, seq, sname, req, is_won, fold, rot in STAGE_SEED:
    if sname in existing_stage_names:
        continue
    stage_rows.append(Row(
        crm_stage_id           = sid,
        sequence               = seq,
        rotting_threshold_days = rot,
        name                   = sname,
        requirements           = req,
        is_won                 = is_won,
        fold                   = fold,
        created_at             = now,
        updated_at             = now,
        dwh_loaded_at          = now,
        dwh_source_db          = SOURCE_DB,
    ))

stage_schema = StructType([
    StructField("crm_stage_id",           IntegerType(),   True),
    StructField("sequence",               IntegerType(),   True),
    StructField("rotting_threshold_days", IntegerType(),   True),
    StructField("name",                   StringType(),    True),
    StructField("requirements",           StringType(),    True),
    StructField("is_won",                 BooleanType(),   True),
    StructField("fold",                   BooleanType(),   True),
    StructField("created_at",             TimestampType(), True),
    StructField("updated_at",             TimestampType(), True),
    StructField("dwh_loaded_at",          TimestampType(), True),
    StructField("dwh_source_db",          StringType(),    True),
])

if stage_rows:
    append_delta(spark.createDataFrame(stage_rows, schema=stage_schema), "crm_stage")
else:
    print("  ⏭   crm_stage  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 2. CRM TEAM  (3 fixed teams — seed once)
# ═══════════════════════════════════════════════════════════════
TEAM_SEED = [
    (1, "Direct Sales",  1, 50000.0, True,  True,  False, False),
    (2, "Online Sales",  2, 80000.0, True,  True,  True,  False),
    (3, "Field Sales",   3, 30000.0, True,  False, True,  True),
]
# (id, name, seq, invoiced_target, active, use_leads, use_opps, assign_optout)

TEAMS = {row[0]: row[1] for row in TEAM_SEED}

existing_team_names = existing_names("crm_team")
team_rows = []
for tid, tname, seq, inv_tgt, active, use_l, use_o, optout in TEAM_SEED:
    if tname in existing_team_names:
        continue
    user_id = random.randint(1, 10)
    team_rows.append(Row(
        crm_team_id                  = tid,
        company_id                   = COMPANY_ID,
        company_name                 = COMPANY,
        user_id                      = user_id,
        user_name                    = f"User_{user_id}",
        sequence                     = seq,
        name                         = tname,
        assignment_domain            = None,
        lead_properties_definition   = None,
        invoiced_target              = inv_tgt,
        active                       = active,
        use_leads                    = use_l,
        use_opportunities            = use_o,
        assignment_optout            = optout,
        created_at                   = now,
        updated_at                   = now,
        dwh_loaded_at                = now,
        dwh_source_db                = SOURCE_DB,
    ))

team_schema = StructType([
    StructField("crm_team_id",                IntegerType(),   True),
    StructField("company_id",                 IntegerType(),   True),
    StructField("company_name",               StringType(),    True),
    StructField("user_id",                    IntegerType(),   True),
    StructField("user_name",                  StringType(),    True),
    StructField("sequence",                   IntegerType(),   True),
    StructField("name",                       StringType(),    True),
    StructField("assignment_domain",          StringType(),    True),
    StructField("lead_properties_definition", StringType(),    True),
    StructField("invoiced_target",            DoubleType(),    True),
    StructField("active",                     BooleanType(),   True),
    StructField("use_leads",                  BooleanType(),   True),
    StructField("use_opportunities",          BooleanType(),   True),
    StructField("assignment_optout",          BooleanType(),   True),
    StructField("created_at",                 TimestampType(), True),
    StructField("updated_at",                 TimestampType(), True),
    StructField("dwh_loaded_at",              TimestampType(), True),
    StructField("dwh_source_db",              StringType(),    True),
])

if team_rows:
    append_delta(spark.createDataFrame(team_rows, schema=team_schema), "crm_team")
else:
    print("  ⏭   crm_team  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 3. CRM LOST REASON  (4 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
LOST_REASON_SEED = [
    (1, "Price"),
    (2, "No budget"),
    (3, "No need"),
    (4, "Other"),
]
LOST_REASONS = {row[0]: row[1] for row in LOST_REASON_SEED}

existing_lr_names = existing_names("crm_lost_reason")
lr_rows = []
for lrid, lname in LOST_REASON_SEED:
    if lname in existing_lr_names:
        continue
    lr_rows.append(Row(
        crm_lost_reason_id = lrid,
        name               = lname,
        active             = True,
        created_at         = now,
        updated_at         = now,
        dwh_loaded_at      = now,
        dwh_source_db      = SOURCE_DB,
    ))

lr_schema = StructType([
    StructField("crm_lost_reason_id", IntegerType(),   True),
    StructField("name",               StringType(),    True),
    StructField("active",             BooleanType(),   True),
    StructField("created_at",         TimestampType(), True),
    StructField("updated_at",         TimestampType(), True),
    StructField("dwh_loaded_at",      TimestampType(), True),
    StructField("dwh_source_db",      StringType(),    True),
])

if lr_rows:
    append_delta(spark.createDataFrame(lr_rows, schema=lr_schema), "crm_lost_reason")
else:
    print("  ⏭   crm_lost_reason  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 4. UTM CAMPAIGN  (4 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
CAMPAIGN_SEED = [
    (1, "Summer Sale",    "Summer Sale 2025"),
    (2, "Q1 Awareness",  "Q1 Brand Awareness"),
    (3, "Retention",     "Customer Retention Drive"),
    (4, "Launch",        "New Product Launch"),
]
CAMPAIGNS = {row[0]: row[1] for row in CAMPAIGN_SEED}

existing_camp_names = existing_names("utm_campaign")
camp_rows = []
for cid, cname, ctitle in CAMPAIGN_SEED:
    if cname in existing_camp_names:
        continue
    uid = random.randint(1, 10)
    camp_rows.append(Row(
        utm_campaign_id  = cid,
        company_id       = COMPANY_ID,
        company_name     = COMPANY,
        user_id          = uid,
        user_name        = f"User_{uid}",
        stage_id         = random.choice(list(STAGES.keys())),
        stage_name       = random.choice(list(STAGES.values())),
        name             = cname,
        title            = ctitle,
        active           = True,
        is_auto_campaign = False,
        created_at       = now,
        updated_at       = now,
        dwh_loaded_at    = now,
        dwh_source_db    = SOURCE_DB,
    ))

camp_schema = StructType([
    StructField("utm_campaign_id",  IntegerType(),   True),
    StructField("company_id",       IntegerType(),   True),
    StructField("company_name",     StringType(),    True),
    StructField("user_id",          IntegerType(),   True),
    StructField("user_name",        StringType(),    True),
    StructField("stage_id",         IntegerType(),   True),
    StructField("stage_name",       StringType(),    True),
    StructField("name",             StringType(),    True),
    StructField("title",            StringType(),    True),
    StructField("active",           BooleanType(),   True),
    StructField("is_auto_campaign", BooleanType(),   True),
    StructField("created_at",       TimestampType(), True),
    StructField("updated_at",       TimestampType(), True),
    StructField("dwh_loaded_at",    TimestampType(), True),
    StructField("dwh_source_db",    StringType(),    True),
])

if camp_rows:
    append_delta(spark.createDataFrame(camp_rows, schema=camp_schema), "utm_campaign")
else:
    print("  ⏭   utm_campaign  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 5. UTM SOURCE  (5 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
SOURCE_SEED = [
    (1, "Google"),
    (2, "Facebook"),
    (3, "LinkedIn"),
    (4, "Email"),
    (5, "Referral"),
]
SOURCES = {row[0]: row[1] for row in SOURCE_SEED}

existing_src_names = existing_names("utm_source")
src_rows = []
for sid, sname in SOURCE_SEED:
    if sname in existing_src_names:
        continue
    src_rows.append(Row(
        utm_source_id = sid,
        name          = sname,
        created_at    = now,
        updated_at    = now,
        dwh_loaded_at = now,
        dwh_source_db = SOURCE_DB,
    ))

src_schema = StructType([
    StructField("utm_source_id", IntegerType(),   True),
    StructField("name",          StringType(),    True),
    StructField("created_at",    TimestampType(), True),
    StructField("updated_at",    TimestampType(), True),
    StructField("dwh_loaded_at", TimestampType(), True),
    StructField("dwh_source_db", StringType(),    True),
])

if src_rows:
    append_delta(spark.createDataFrame(src_rows, schema=src_schema), "utm_source")
else:
    print("  ⏭   utm_source  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 6. UTM MEDIUM  (4 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
MEDIUM_SEED = [
    (1, "CPC"),
    (2, "Organic"),
    (3, "Social"),
    (4, "Email"),
]
MEDIUMS = {row[0]: row[1] for row in MEDIUM_SEED}

existing_med_names = existing_names("utm_medium")
med_rows = []
for mid, mname in MEDIUM_SEED:
    if mname in existing_med_names:
        continue
    med_rows.append(Row(
        utm_medium_id = mid,
        name          = mname,
        active        = True,
        created_at    = now,
        updated_at    = now,
        dwh_loaded_at = now,
        dwh_source_db = SOURCE_DB,
    ))

med_schema = StructType([
    StructField("utm_medium_id", IntegerType(),   True),
    StructField("name",          StringType(),    True),
    StructField("active",        BooleanType(),   True),
    StructField("created_at",    TimestampType(), True),
    StructField("updated_at",    TimestampType(), True),
    StructField("dwh_loaded_at", TimestampType(), True),
    StructField("dwh_source_db", StringType(),    True),
])

if med_rows:
    append_delta(spark.createDataFrame(med_rows, schema=med_schema), "utm_medium")
else:
    print("  ⏭   utm_medium  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 7. CRM LEAD  — 300 NEW rows every run
# ═══════════════════════════════════════════════════════════════

lead_id_start = max_id("crm_lead", "crm_lead_id") + 1
print(f"\nStarting crm_lead_id from : {lead_id_start}")

# ── pools ──────────────────────────────────────────────────────
LEAD_TYPES   = (["lead"] * 40) + (["opportunity"] * 60)
PRIORITIES   = (["0"] * 50) + (["1"] * 25) + (["2"] * 15) + (["3"] * 10)
PHONE_STATES = (["correct"] * 60) + (["incorrect"] * 15) + ([None] * 25)
EMAIL_STATES = (["correct"] * 65) + (["incorrect"] * 10) + ([None] * 25)
CITIES       = (["Cairo"] * 35 + ["Alex"] * 20 + ["Giza"] * 20 +
                ["Mansoura"] * 10 + ["Tanta"] * 8 + ["Aswan"] * 7)
STREETS      = ["El-Nasr St", "Tahrir Sq", "Corniche Rd", "Port Said St",
                "El-Horreya Ave", "El-Galaa St", None, None]
FUNCTIONS    = ["Sales Manager", "CEO", "Procurement Officer", "IT Director",
                "Finance Manager", None, None, None]

# stage → (prob_min, prob_max, won_status)
STAGE_LOGIC = {
    1: (0.05, 0.20, "pending"),   # New
    2: (0.20, 0.40, "pending"),   # Qualified
    3: (0.40, 0.70, "pending"),   # Proposition
    4: (0.90, 1.00, "won"),       # Won
    5: (0.00, 0.15, "lost"),      # Lost
}

lead_rows = []
lead_id   = lead_id_start

for _ in range(300):

    # FK references
    stage_id   = random.choice(list(STAGES.keys()))
    stage_name = STAGES[stage_id]
    team_id    = random.choice(list(TEAMS.keys()))
    team_name  = TEAMS[team_id]
    camp_id    = random.choice(list(CAMPAIGNS.keys()))
    src_id     = random.choice(list(SOURCES.keys()))
    med_id     = random.choice(list(MEDIUMS.keys()))
    user_id    = random.randint(1, 10)
    partner_id = random.randint(1, 50)

    # lost reason — only for Lost stage
    if stage_id == 5:
        lr_id   = random.choice(list(LOST_REASONS.keys()))
        lr_name = LOST_REASONS[lr_id]
    else:
        lr_id, lr_name = None, None

    lead_type = random.choice(LEAD_TYPES)
    priority  = random.choice(PRIORITIES)

    # revenue
    exp_rev   = round(random.uniform(1_000, 100_000), 2)
    prob_min, prob_max, won_status = STAGE_LOGIC[stage_id]
    probability      = round(random.uniform(prob_min, prob_max), 4)
    auto_prob        = round(probability * random.uniform(0.8, 1.1), 4)
    prorated_rev     = round(exp_rev * probability, 2)
    recurring        = round(exp_rev * random.uniform(0.05, 0.20), 2) if random.random() > 0.8 else 0.0
    rec_monthly      = round(recurring / 12, 2)
    rec_prorated     = round(recurring * probability, 2)
    rec_monthly_pro  = round(rec_monthly * probability, 2)

    # dates
    date_open        = rand_date()
    date_last_stage  = date_open + timedelta(days=random.randint(0, 15))
    date_deadline    = maybe(date_open + timedelta(days=random.randint(10, 90)), 0.25)
    date_conversion  = maybe(
        date_open + timedelta(days=random.randint(1, 30)),
        0.40 if lead_type == "opportunity" else 1.0   # only opportunities
    )

    # close date only for Won / Lost
    if won_status in ("won", "lost"):
        date_closed  = date_open + timedelta(days=random.randint(5, 60))
        day_close    = (date_closed - date_open).days
    else:
        date_closed  = None
        day_close    = None

    day_open = (now - date_open).days

    # active flag — lost leads mostly inactive
    if won_status == "lost":
        active = random.random() > 0.7
    else:
        active = True

    city     = random.choice(CITIES)
    zip_code = str(random.randint(10000, 99999))

    email    = f"contact_{lead_id}@company{random.randint(1,999)}.com"
    phone    = f"010{random.randint(10000000, 99999999)}"

    lead_rows.append(Row(
        crm_lead_id                          = lead_id,
        campaign_id                          = camp_id,
        campaign_name                        = CAMPAIGNS[camp_id],
        source_id                            = src_id,
        source_name                          = SOURCES[src_id],
        medium_id                            = med_id,
        medium_name                          = MEDIUMS[med_id],
        user_id                              = user_id,
        user_name                            = f"User_{user_id}",
        team_id                              = team_id,
        team_name                            = team_name,
        company_id                           = COMPANY_ID,
        company_name                         = COMPANY,
        stage_id                             = stage_id,
        stage_name                           = stage_name,
        partner_id                           = partner_id,
        partner_name                         = f"Partner_{partner_id:03d}",
        country_id                           = COUNTRY_ID,
        country_name                         = COUNTRY,
        lost_reason_id                       = lr_id,
        lost_reason_name                     = lr_name,
        name                                 = f"Lead {lead_id} - {stage_name}",
        type                                 = lead_type,
        priority                             = priority,
        contact_name                         = f"Contact_{lead_id}",
        function                             = random.choice(FUNCTIONS),
        email_from                           = email,
        email_normalized                     = email.lower(),
        email_cc                             = maybe(f"cc_{lead_id}@mail.com", 0.7),
        phone                                = phone,
        phone_sanitized                      = phone,
        phone_state                          = random.choice(PHONE_STATES),
        email_state                          = random.choice(EMAIL_STATES),
        website                              = maybe(f"https://company{random.randint(1,999)}.com", 0.6),
        street                               = random.choice(STREETS),
        street2                              = maybe(f"Floor {random.randint(1,20)}", 0.7),
        zip                                  = zip_code,
        city                                 = city,
        won_status                           = won_status,
        description                          = maybe(f"Lead note {lead_id}", 0.5),
        lead_properties                      = None,
        referred                             = maybe(f"Ref_{random.randint(100,999)}", 0.8),
        expected_revenue                     = exp_rev,
        prorated_revenue                     = prorated_rev,
        recurring_revenue                    = recurring,
        recurring_revenue_monthly            = rec_monthly,
        recurring_revenue_monthly_prorated   = rec_monthly_pro,
        recurring_revenue_prorated           = rec_prorated,
        probability                          = probability,
        automated_probability                = min(auto_prob, 1.0),
        day_open                             = float(day_open),
        day_close                            = float(day_close) if day_close is not None else None,
        active                               = active,
        iap_enrich_done                      = random.random() < 0.3,
        date_deadline                        = date_deadline,
        date_closed                          = date_closed,
        date_open                            = date_open,
        date_last_stage_update               = date_last_stage,
        date_conversion                      = date_conversion,
        created_at                           = date_open,
        updated_at                           = date_open,
        dwh_loaded_at                        = now,
        dwh_source_db                        = SOURCE_DB,
    ))

    lead_id += 1


lead_schema = StructType([
    StructField("crm_lead_id",                          IntegerType(),   True),
    StructField("campaign_id",                          IntegerType(),   True),
    StructField("campaign_name",                        StringType(),    True),
    StructField("source_id",                            IntegerType(),   True),
    StructField("source_name",                          StringType(),    True),
    StructField("medium_id",                            IntegerType(),   True),
    StructField("medium_name",                          StringType(),    True),
    StructField("user_id",                              IntegerType(),   True),
    StructField("user_name",                            StringType(),    True),
    StructField("team_id",                              IntegerType(),   True),
    StructField("team_name",                            StringType(),    True),
    StructField("company_id",                           IntegerType(),   True),
    StructField("company_name",                         StringType(),    True),
    StructField("stage_id",                             IntegerType(),   True),
    StructField("stage_name",                           StringType(),    True),
    StructField("partner_id",                           IntegerType(),   True),
    StructField("partner_name",                         StringType(),    True),
    StructField("country_id",                           IntegerType(),   True),
    StructField("country_name",                         StringType(),    True),
    StructField("lost_reason_id",                       IntegerType(),   True),
    StructField("lost_reason_name",                     StringType(),    True),
    StructField("name",                                 StringType(),    True),
    StructField("type",                                 StringType(),    True),
    StructField("priority",                             StringType(),    True),
    StructField("contact_name",                         StringType(),    True),
    StructField("function",                             StringType(),    True),
    StructField("email_from",                           StringType(),    True),
    StructField("email_normalized",                     StringType(),    True),
    StructField("email_cc",                             StringType(),    True),
    StructField("phone",                                StringType(),    True),
    StructField("phone_sanitized",                      StringType(),    True),
    StructField("phone_state",                          StringType(),    True),
    StructField("email_state",                          StringType(),    True),
    StructField("website",                              StringType(),    True),
    StructField("street",                               StringType(),    True),
    StructField("street2",                              StringType(),    True),
    StructField("zip",                                  StringType(),    True),
    StructField("city",                                 StringType(),    True),
    StructField("won_status",                           StringType(),    True),
    StructField("description",                          StringType(),    True),
    StructField("lead_properties",                      StringType(),    True),
    StructField("referred",                             StringType(),    True),
    StructField("expected_revenue",                     DoubleType(),    True),
    StructField("prorated_revenue",                     DoubleType(),    True),
    StructField("recurring_revenue",                    DoubleType(),    True),
    StructField("recurring_revenue_monthly",            DoubleType(),    True),
    StructField("recurring_revenue_monthly_prorated",   DoubleType(),    True),
    StructField("recurring_revenue_prorated",           DoubleType(),    True),
    StructField("probability",                          DoubleType(),    True),
    StructField("automated_probability",                DoubleType(),    True),
    StructField("day_open",                             DoubleType(),    True),
    StructField("day_close",                            DoubleType(),    True),
    StructField("active",                               BooleanType(),   True),
    StructField("iap_enrich_done",                      BooleanType(),   True),
    StructField("date_deadline",                        TimestampType(), True),
    StructField("date_closed",                          TimestampType(), True),
    StructField("date_open",                            TimestampType(), True),
    StructField("date_last_stage_update",               TimestampType(), True),
    StructField("date_conversion",                      TimestampType(), True),
    StructField("created_at",                           TimestampType(), True),
    StructField("updated_at",                           TimestampType(), True),
    StructField("dwh_loaded_at",                        TimestampType(), True),
    StructField("dwh_source_db",                        StringType(),    True),
])

df_leads = spark.createDataFrame(lead_rows, schema=lead_schema)
append_delta(df_leads, "crm_lead")

print("\n✅  DONE — All 7 CRM Delta tables written to smart_odoo.silver")